# Práctica 3 — Bases de datos relacionales

**Estudiante:** Brailin Leonardo Méndez Guillermo  
**Objetivo:** Modelar clientes, productos y ventas; aplicar consultas, JOIN, agregaciones y transacciones con SQLite.


In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(":memory:")
conn.executescript("""
PRAGMA foreign_keys = ON;
CREATE TABLE clientes(id INTEGER PRIMARY KEY, nombre TEXT NOT NULL, ciudad TEXT NOT NULL);
CREATE TABLE productos(id INTEGER PRIMARY KEY, nombre TEXT NOT NULL, precio REAL NOT NULL CHECK(precio > 0));
CREATE TABLE ventas(
  id INTEGER PRIMARY KEY,
  cliente_id INTEGER NOT NULL REFERENCES clientes(id),
  producto_id INTEGER NOT NULL REFERENCES productos(id),
  cantidad INTEGER NOT NULL CHECK(cantidad > 0),
  fecha TEXT NOT NULL
);
INSERT INTO clientes VALUES (1,'Ana','Santo Domingo'),(2,'Luis','Santiago'),(3,'Marta','Santo Domingo');
INSERT INTO productos VALUES (1,'Laptop',45000),(2,'Mouse',800),(3,'Teclado',1500);
INSERT INTO ventas VALUES
 (1,1,1,1,'2026-10-01'),(2,1,2,2,'2026-10-02'),
 (3,2,3,1,'2026-10-03'),(4,3,2,3,'2026-10-04');
""")
pd.read_sql_query("SELECT * FROM ventas", conn)


In [ ]:
# JOIN: detalle legible de ventas
consulta = """
SELECT v.id, c.nombre AS cliente, c.ciudad, p.nombre AS producto,
       v.cantidad, p.precio, ROUND(v.cantidad*p.precio, 2) AS total, v.fecha
FROM ventas v
JOIN clientes c ON c.id=v.cliente_id
JOIN productos p ON p.id=v.producto_id
ORDER BY v.fecha;
"""
pd.read_sql_query(consulta, conn)


In [ ]:
# Agregación: gasto por cliente
pd.read_sql_query("""
SELECT c.nombre, COUNT(v.id) AS operaciones,
       ROUND(SUM(v.cantidad*p.precio),2) AS monto_total
FROM clientes c
JOIN ventas v ON v.cliente_id=c.id
JOIN productos p ON p.id=v.producto_id
GROUP BY c.id, c.nombre
ORDER BY monto_total DESC;
""", conn)


In [ ]:
# Filtros, INSERT, UPDATE y DELETE dentro de una transacción
with conn:
    conn.execute("INSERT INTO clientes VALUES (4, ?, ?)", ("Carlos", "La Vega"))
    conn.execute("UPDATE productos SET precio=? WHERE nombre=?", (900, "Mouse"))
    conn.execute("DELETE FROM ventas WHERE id=?", (4,))

print(pd.read_sql_query("SELECT * FROM clientes WHERE ciudad='Santo Domingo'", conn))
print(pd.read_sql_query("SELECT * FROM productos WHERE precio > 1000 ORDER BY precio DESC", conn))
